# Import các thư viện cần thiết

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
import time

# Question 1: Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

In [48]:
orders = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\orders.csv")
print(orders.info())
orders

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   order_id        646945 non-null  int64 
 1   order_date      646945 non-null  object
 2   customer_id     646945 non-null  int64 
 3   zip             646945 non-null  int64 
 4   order_status    646945 non-null  object
 5   payment_method  646945 non-null  object
 6   device_type     646945 non-null  object
 7   order_source    646945 non-null  object
dtypes: int64(3), object(5)
memory usage: 39.5+ MB
None


,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign
...,...,...,...,...,...,...,...,...
646940,834372,2022-12-31,19490,33907,delivered,credit_card,mobile,email_campaign
646941,834377,2022-12-31,73046,37091,delivered,credit_card,mobile,referral
646942,834387,2022-12-31,107723,80516,delivered,credit_card,mobile,email_campaign
646943,834392,2022-12-31,139431,93510,delivered,paypal,desktop,direct


In [49]:
# Chuyển đổi kiểu dữ liệu
orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    errors="coerce"
)

# Bỏ dòng ngày lỗi
orders = orders.dropna(subset=["order_date"])


# Chỉ giữ những khách hàng có trên 1 đớn hàng
customer_orders = orders["customer_id"].value_counts()

multi_customers = customer_orders[
    customer_orders > 1
].index

orders = orders[
    orders["customer_id"].isin(multi_customers)
]

# Sắp xếp dữ liệu
orders = orders.sort_values(
    ["customer_id", "order_date"]
).reset_index(drop=True)

# Lấy ngày mua trước đó của cùng khách hàng
orders["prev_order_date"] = orders.groupby(
    "customer_id"
)["order_date"].shift(1)

orders["prev_order_date"] = pd.to_datetime(
    orders["prev_order_date"],
    errors="coerce"
)

# Tính số ngày giữa 2 lần mua liên tiếp
orders["gap_days"] = (
    orders["order_date"] - orders["prev_order_date"]
).dt.days

# median
median_gap = orders["gap_days"].dropna().median()

print("Median inter-order gap =", median_gap)

Median inter-order gap = 144.0


# Question 2: Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?

In [50]:
products = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\products.csv")
print(products.info())
products

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    2412 non-null   int64  
 1   product_name  2412 non-null   object 
 2   category      2412 non-null   object 
 3   segment       2412 non-null   object 
 4   size          2412 non-null   object 
 5   color         2412 non-null   object 
 6   price         2412 non-null   float64
 7   cogs          2412 non-null   float64
dtypes: float64(2), int64(1), object(5)
memory usage: 150.9+ KB
None


,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406
...,...,...,...,...,...,...,...,...
2407,1260,VietMode MP-28,Casual,Activewear,S,red,4603.340000,2553.933032
2408,1261,VietMode MP-29,Casual,Activewear,M,black,5983.876433,4653.660702
2409,1262,VietMode MP-30,Casual,Activewear,L,orange,5983.876433,5684.682611
2410,1263,VietMode MP-31,Casual,Activewear,XL,blue,5984.370000,5685.151500


In [51]:
# Tính Gross Margin
# (price - cogs) / price
products["gross_margin"] = (
    (products["price"] - products["cogs"]) /
    products["price"]
)

# Tính trung bình theo Segment
result = products.groupby("segment")[
    "gross_margin"
].mean().reset_index()

# sắp xếp giảm dần
result = result.sort_values(
    by="gross_margin",
    ascending=False
).reset_index(drop=True)

print("=" * 50)
print(result)
print("=" * 50)

best_segment = result.loc[0, "segment"]
best_value = result.loc[0, "gross_margin"]

print("Segment cao nhất:", best_segment)
print("Average Gross Margin:", round(best_value * 100, 2), "%")

       segment  gross_margin
0     Standard      0.313442
1      Premium      0.285377
2  All-weather      0.284176
3   Activewear      0.265600
4  Performance      0.263650
5     Balanced      0.258038
6       Trendy      0.240758
7     Everyday      0.236343
Segment cao nhất: Standard
Average Gross Margin: 31.34 %


# Question 3: Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [52]:
returns = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\returns.csv")
print(returns.info())
returns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   return_id        39939 non-null  object 
 1   order_id         39939 non-null  int64  
 2   product_id       39939 non-null  int64  
 3   return_date      39939 non-null  object 
 4   return_reason    39939 non-null  object 
 5   return_quantity  39939 non-null  int64  
 6   refund_amount    39939 non-null  float64
dtypes: float64(1), int64(3), object(3)
memory usage: 2.1+ MB
None


,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76
...,...,...,...,...,...,...,...
39934,RET-051470,832867,653,2022-12-27,wrong_size,3,24741.62
39935,RET-051471,832890,792,2022-12-30,late_delivery,1,560.50
39936,RET-051481,833005,449,2022-12-31,defective,1,10002.55
39937,RET-051494,833234,1085,2022-12-28,wrong_size,1,815.57


In [ ]:
products = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\products.csv")
print(products.info())
products

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    2412 non-null   int64  
 1   product_name  2412 non-null   object 
 2   category      2412 non-null   object 
 3   segment       2412 non-null   object 
 4   size          2412 non-null   object 
 5   color         2412 non-null   object 
 6   price         2412 non-null   float64
 7   cogs          2412 non-null   float64
dtypes: float64(2), int64(1), object(5)
memory usage: 150.9+ KB
None


,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406
...,...,...,...,...,...,...,...,...
2407,1260,VietMode MP-28,Casual,Activewear,S,red,4603.340000,2553.933032
2408,1261,VietMode MP-29,Casual,Activewear,M,black,5983.876433,4653.660702
2409,1262,VietMode MP-30,Casual,Activewear,L,orange,5983.876433,5684.682611
2410,1263,VietMode MP-31,Casual,Activewear,XL,blue,5984.370000,5685.151500


In [54]:
# Join 2 bảng theo product_id
df = returns.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left"
)

# Chỉ lấy Streetwear
df = df[df["category"] == "Streetwear"]

# Đếm lý do trả hàng
result = df["return_reason"].value_counts()

print(result)
print("Đáp án:", result.idxmax())

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64
Đáp án: wrong_size


# Question 4: Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [55]:
web_traffics = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\web_traffic.csv")
print(web_traffics.info())
web_traffics

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3652 entries, 0 to 3651
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      3652 non-null   object 
 1   sessions                  3652 non-null   int64  
 2   unique_visitors           3652 non-null   int64  
 3   page_views                3652 non-null   int64  
 4   bounce_rate               3652 non-null   float64
 5   avg_session_duration_sec  3652 non-null   float64
 6   traffic_source            3652 non-null   object 
dtypes: float64(2), int64(3), object(2)
memory usage: 199.8+ KB
None


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct
3,2013-01-04,9973,8063,53078,0.00562,151.8,direct
4,2013-01-05,10223,7882,36790,0.00525,168.6,referral
...,...,...,...,...,...,...,...
3647,2022-12-27,17416,13150,62527,0.00506,252.4,organic_search
3648,2022-12-28,21071,15979,67456,0.00560,177.3,organic_search
3649,2022-12-29,20884,14640,82155,0.00522,165.6,direct
3650,2022-12-30,17679,13713,79308,0.00350,183.8,email_campaign


In [56]:
result = web_traffics.groupby("traffic_source")[
    "bounce_rate"
].mean().sort_values()

print(result)
print("Đáp án:", result.index[0])

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64
Đáp án: email_campaign


# Question 5: Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

In [57]:
order_items = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\order_items.csv")
print(order_items.info())
order_items

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  int64  
 1   product_id       714669 non-null  int64  
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         276316 non-null  object 
 6   promo_id_2       206 non-null     object 
dtypes: float64(2), int64(3), object(2)
memory usage: 38.2+ MB
None


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6500\1613078720.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\order_items.csv")


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN
3,4,635,5,10639.25,0.0,NaN,NaN
4,6,1935,1,1597.84,0.0,NaN,NaN
...,...,...,...,...,...,...,...
714664,834372,690,8,4473.92,0.0,NaN,NaN
714665,834377,1995,7,5250.79,0.0,NaN,NaN
714666,834387,2331,8,7389.06,0.0,NaN,NaN
714667,834392,1115,5,4767.33,0.0,NaN,NaN


In [ ]:
# Tổng số dòng
total_rows = len(order_items)

# Chuẩn hóa null giả
order_items["promo_id"] = order_items["promo_id"].replace(
    ["", "nan", "NaN", "None"],
    pd.NA
)

order_items["promo_id_2"] = order_items["promo_id_2"].replace(
    ["", "nan", "NaN", "None"],
    pd.NA
)

# Có khuyến mãi nếu promo_id hoặc promo_id_2 có giá trị
promo_rows = order_items[
    order_items["promo_id"].notna() |
    order_items["promo_id_2"].notna()
]

promo_count = len(promo_rows)

# 4. Tính %
pct = (promo_count / total_rows) * 100

print("Tổng số dòng:", total_rows)
print("Dòng có khuyến mãi:", promo_count)
print("Tỷ lệ:", round(pct, 2), "%")

Tổng số dòng: 714669
Dòng có khuyến mãi: 276316
Tỷ lệ: 38.66 %


# Question 6: Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

In [59]:
orders = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\orders.csv")
print(orders.info())
orders

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   order_id        646945 non-null  int64 
 1   order_date      646945 non-null  object
 2   customer_id     646945 non-null  int64 
 3   zip             646945 non-null  int64 
 4   order_status    646945 non-null  object
 5   payment_method  646945 non-null  object
 6   device_type     646945 non-null  object
 7   order_source    646945 non-null  object
dtypes: int64(3), object(5)
memory usage: 39.5+ MB
None


,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign
...,...,...,...,...,...,...,...,...
646940,834372,2022-12-31,19490,33907,delivered,credit_card,mobile,email_campaign
646941,834377,2022-12-31,73046,37091,delivered,credit_card,mobile,referral
646942,834387,2022-12-31,107723,80516,delivered,credit_card,mobile,email_campaign
646943,834392,2022-12-31,139431,93510,delivered,paypal,desktop,direct


In [60]:
customers = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\customers.csv")
print(customers.info())
customers

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   customer_id          121930 non-null  int64 
 1   zip                  121930 non-null  int64 
 2   city                 121930 non-null  object
 3   signup_date          121930 non-null  object
 4   gender               121930 non-null  object
 5   age_group            121930 non-null  object
 6   acquisition_channel  121930 non-null  object
dtypes: int64(2), object(5)
memory usage: 6.5+ MB
None


,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search
3,4,15201,Hai Phong,2017-11-29,Male,35-44,referral
4,5,15201,Hai Phong,2022-09-23,Male,55+,organic_search
...,...,...,...,...,...,...,...
121925,157556,59936,Vung Tau,2016-03-03,Female,18-24,direct
121926,157557,59936,Vung Tau,2021-05-11,Female,45-54,social_media
121927,157558,59936,Vung Tau,2017-02-27,Female,25-34,referral
121928,157561,59937,Buon Ma Thuot,2018-10-15,Non-binary,45-54,paid_search


In [61]:
df = orders.merge(
    customers[["customer_id","age_group"]],
    on="customer_id",
    how="left"
)

# bỏ null
df = df[df["age_group"].notna()]
df = df[df["age_group"] != "Unknown"]

# tổng đơn / số khách
orders_count = df.groupby("age_group")["order_id"].count()
cust_count = df.groupby("age_group")["customer_id"].nunique()

result = (orders_count / cust_count).sort_values(ascending=False)

print(result)
print("Đáp án:", result.index[0])

age_group
55+      7.268731
45-54    7.220264
35-44    7.206159
25-34    7.112230
18-24    7.068577
dtype: float64
Đáp án: 55+


# Question 7: Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?

In [62]:
orders = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\orders.csv")
print(orders.info())
orders

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   order_id        646945 non-null  int64 
 1   order_date      646945 non-null  object
 2   customer_id     646945 non-null  int64 
 3   zip             646945 non-null  int64 
 4   order_status    646945 non-null  object
 5   payment_method  646945 non-null  object
 6   device_type     646945 non-null  object
 7   order_source    646945 non-null  object
dtypes: int64(3), object(5)
memory usage: 39.5+ MB
None


,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign
...,...,...,...,...,...,...,...,...
646940,834372,2022-12-31,19490,33907,delivered,credit_card,mobile,email_campaign
646941,834377,2022-12-31,73046,37091,delivered,credit_card,mobile,referral
646942,834387,2022-12-31,107723,80516,delivered,credit_card,mobile,email_campaign
646943,834392,2022-12-31,139431,93510,delivered,paypal,desktop,direct


In [63]:
payments = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\payments.csv")
print(payments.info())
payments

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   order_id        646945 non-null  int64  
 1   payment_method  646945 non-null  object 
 2   payment_value   646945 non-null  float64
 3   installments    646945 non-null  int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 19.7+ MB
None


,order_id,payment_method,payment_value,installments
0,1,credit_card,7967.54,3
1,2,cod,71163.75,1
2,3,credit_card,33660.99,3
3,4,credit_card,53196.25,3
4,6,paypal,1597.84,1
...,...,...,...,...
646940,834372,credit_card,35791.36,3
646941,834377,credit_card,36755.53,3
646942,834387,credit_card,59112.48,1
646943,834392,paypal,23836.65,1


In [64]:
geo = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\geography.csv")
print(geo.info())
geo

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39948 entries, 0 to 39947
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   zip       39948 non-null  int64 
 1   city      39948 non-null  object
 2   region    39948 non-null  object
 3   district  39948 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.2+ MB
None


,zip,city,region,district
0,15201,Hai Phong,East,District #13
1,15202,Phu Ly,East,District #13
2,15203,Viet Tri,East,District #13
3,15204,Bac Giang,East,District #13
4,15205,Bac Giang,East,District #13
...,...,...,...,...
39943,59933,Pleiku,West,District #33
39944,59934,Soc Trang,West,District #33
39945,59935,Rach Gia,West,District #33
39946,59936,Vung Tau,West,District #33


In [65]:
df = orders.merge(
    geo[["zip","region"]],
    on="zip",
    how="left"
).merge(
    payments[["order_id","payment_value"]],
    on="order_id",
    how="left"
)

result = df.groupby("region")[
    "payment_value"
].sum().sort_values(ascending=False)

print(result)
print("Đáp án:", result.index[0])

region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: payment_value, dtype: float64
Đáp án: East


# Question 8: Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?

In [66]:
orders = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\orders.csv")
print(orders.info())
orders

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   order_id        646945 non-null  int64 
 1   order_date      646945 non-null  object
 2   customer_id     646945 non-null  int64 
 3   zip             646945 non-null  int64 
 4   order_status    646945 non-null  object
 5   payment_method  646945 non-null  object
 6   device_type     646945 non-null  object
 7   order_source    646945 non-null  object
dtypes: int64(3), object(5)
memory usage: 39.5+ MB
None


,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign
...,...,...,...,...,...,...,...,...
646940,834372,2022-12-31,19490,33907,delivered,credit_card,mobile,email_campaign
646941,834377,2022-12-31,73046,37091,delivered,credit_card,mobile,referral
646942,834387,2022-12-31,107723,80516,delivered,credit_card,mobile,email_campaign
646943,834392,2022-12-31,139431,93510,delivered,paypal,desktop,direct


In [67]:
orders = orders[
    orders["order_status"].isin(
        ["cancelled", "canceled"]
    )
]

result = orders["payment_method"].value_counts()

print(result)
print("Đáp án:", result.idxmax())

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64
Đáp án: credit_card


# Question 9: Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?

In [68]:
returns = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\returns.csv")
print(returns.info())
returns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   return_id        39939 non-null  object 
 1   order_id         39939 non-null  int64  
 2   product_id       39939 non-null  int64  
 3   return_date      39939 non-null  object 
 4   return_reason    39939 non-null  object 
 5   return_quantity  39939 non-null  int64  
 6   refund_amount    39939 non-null  float64
dtypes: float64(1), int64(3), object(3)
memory usage: 2.1+ MB
None


,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76
...,...,...,...,...,...,...,...
39934,RET-051470,832867,653,2022-12-27,wrong_size,3,24741.62
39935,RET-051471,832890,792,2022-12-30,late_delivery,1,560.50
39936,RET-051481,833005,449,2022-12-31,defective,1,10002.55
39937,RET-051494,833234,1085,2022-12-28,wrong_size,1,815.57


In [69]:
products = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\products.csv")
print(products.info())
products

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    2412 non-null   int64  
 1   product_name  2412 non-null   object 
 2   category      2412 non-null   object 
 3   segment       2412 non-null   object 
 4   size          2412 non-null   object 
 5   color         2412 non-null   object 
 6   price         2412 non-null   float64
 7   cogs          2412 non-null   float64
dtypes: float64(2), int64(1), object(5)
memory usage: 150.9+ KB
None


,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406
...,...,...,...,...,...,...,...,...
2407,1260,VietMode MP-28,Casual,Activewear,S,red,4603.340000,2553.933032
2408,1261,VietMode MP-29,Casual,Activewear,M,black,5983.876433,4653.660702
2409,1262,VietMode MP-30,Casual,Activewear,L,orange,5983.876433,5684.682611
2410,1263,VietMode MP-31,Casual,Activewear,XL,blue,5984.370000,5685.151500


In [70]:
order_items = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\order_items.csv")
print(order_items.info())
order_items

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  int64  
 1   product_id       714669 non-null  int64  
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         276316 non-null  object 
 6   promo_id_2       206 non-null     object 
dtypes: float64(2), int64(3), object(2)
memory usage: 38.2+ MB
None


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6500\1613078720.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\order_items.csv")


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN
3,4,635,5,10639.25,0.0,NaN,NaN
4,6,1935,1,1597.84,0.0,NaN,NaN
...,...,...,...,...,...,...,...
714664,834372,690,8,4473.92,0.0,NaN,NaN
714665,834377,1995,7,5250.79,0.0,NaN,NaN
714666,834387,2331,8,7389.06,0.0,NaN,NaN
714667,834392,1115,5,4767.33,0.0,NaN,NaN


In [71]:
# Tổng dòng order theo size
df = order_items.merge(
    products[["product_id","size"]],
    on="product_id",
    how="left"
)

# Tổng return theo size
rt = returns.merge(
    products[["product_id","size"]],
    on="product_id",
    how="left"
)

df = df[df["size"].isin(["S","M","L","XL"])]
rt = rt[rt["size"].isin(["S","M","L","XL"])]

den = df.groupby("size").size()
num = rt.groupby("size").size()

result = (num / den).sort_values(ascending=False)

print(result)
print("Đáp án:", result.index[0])

size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
dtype: float64
Đáp án: S


# Question 10: Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

In [72]:
payments = pd.read_csv(r"C:\Users\ADMIN\Downloads\DATATHON\DATA\payments.csv")
print(payments.info())
payments

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   order_id        646945 non-null  int64  
 1   payment_method  646945 non-null  object 
 2   payment_value   646945 non-null  float64
 3   installments    646945 non-null  int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 19.7+ MB
None


,order_id,payment_method,payment_value,installments
0,1,credit_card,7967.54,3
1,2,cod,71163.75,1
2,3,credit_card,33660.99,3
3,4,credit_card,53196.25,3
4,6,paypal,1597.84,1
...,...,...,...,...
646940,834372,credit_card,35791.36,3
646941,834377,credit_card,36755.53,3
646942,834387,credit_card,59112.48,1
646943,834392,paypal,23836.65,1


In [73]:
result = payments.groupby("installments")[
    "payment_value"
].mean().sort_values(ascending=False)

print(result)
print("Đáp án:", result.index[0], "kỳ")

installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64
Đáp án: 6 kỳ
